## DashPi Reboot (x2)

The basics of why we're doing this? Why do we have a new Set Up notebook? The Hzeller library was giving me trouble with Raspbian Lite, and after ~15 hours of troubleshooting and me thinking of workarounds, here's what I learned:
1. GPIO 27, Pin 13 on the Raspi, is probably being used for something, and getting in the way of being used to address the RGB Matrix. (Hzeller mentions this in his troubleshooting section, but I was naive and ignored him...)
2. I could have created a custom GPIO Map... but I sure as hell don't want to think about that
3. DietPi was recommended by Hzeller himself. It's much more minimal, and probably will use less GPIO pins.

## Installing DietPi
I just went to the DietPi website and followed their instructions. Remarkably simple, though a tad weird/abrupt:
Just focus on 1) logging in, 2) Setting up Wifi, and 3) Logging in again and letting it start updates.
It will automatically begin all updates on its own.

> sudo dpkg-reconfigure keyboard-configuration

If you try to run the following commands, it should say there's nothing to install/update. (Cuz it already ran it above.)
> sudo apt-get update
> sudo apt full-upgrade 
### This gives power to use the "make" command later
> sudo apt-get install build-essential

## Python3
> sudo apt-get install python3
> sudo apt-get install python3-pip
> sudo apt-get install pypy

# Everything from here on out will be pip3 dependent
# Let's set up some other installers
> sudo pip3 install setuptools
> sudo pip3 install ez_setup
> pip3 install ipython

# On the topic of using *sudo* pip3 -- use it only for installers? Or root necessary things? Avoid otherwise. Messes with things.

> sudo apt-get install tmux # Allows terminal splitting for multi-window work
> sudo apt-get install libatlas-base-dev
> sudo apt install python3-numpy python3-pandas
> sudo apt autoremove

Python 3 and pandas worked after that.

## RPI for the RGB Display only - Installing and setting up HZeller's RGB Library in advance
> sudo apt update
> sudo apt -y install git libgraphicsmagick++-dev libwebp-dev libavcodec-dev libavformat-dev libswscale-dev initramfs-tools
> git clone https://github.com/hzeller/rpi-rgb-led-matrix.git
> cd ~/rpi-rgb-led-matrix/utils
> make led-image-viewer
> make video-viewer (I think this one failed last i tried it)
> cd ~/rpi-rgb-led-matrix/examples-api-use
> make

### Due to an error with the sound module, will be disabling it; type one command at a time
> cat <<EOF | sudo tee /etc/modprobe.d/blacklist-rgb-matrix.conf
## Incase you have trouble finding the '|' character, on the Jelly Comb keyboard, inside DietPi, it's Right Alt + Shift + ` (The top-left-most button)
> blacklist snd_bcm2835
> EOF
> sudo update-initramfs -u
> sudo reboot
> python3

## DashPi doing set up for python3 bindings of Zeller's library
>> sudo apt-get update && sudo apt-get install python3-dev python3-pillow -y
>> cd rpi-rgb-led-matrix
>> make build-python PYTHON=python3
>> sudo make install-python PYTHON=python3


## Setting up the connection to the DashPi Git Repo
### First, we initialize our git repo manually
> git clone https://github.com/FarhadGSRX/DashPi.git DashPi  
> cd DashPi  
> git init

At this point, the git repo is made, and `git pull` can be called in there to update it at any time.

### Crontab and Reboot
#### The reboot process in order:
1. Enter DashPi folder
2. Use root user's crontab to run /home/dietpi/on_reboot.sh
3. That bash file should cd to DashPi folder, and run the following steps:
    3a. Git Pull or Fetch, Incorporate changes or whatever
    3b. Run Dashpi

**sudo crontab -u root -e**

    # @reboot bash /home/dietpi/DashPi/testing_reboot.sh
    @reboot /home/pi/on_reboot.sh
    0 4 * * * sudo reboot

**on_reboot.sh**

    cd /home/dietpi/DashPi
    git fetch --all >> dashpi_reboot_log.txt
    wait
    git reset --hard origin/master >> dashpi_reboot_log.txt
    wait
    pip3 install -r requirements.txt >> dashpi_reboot_log.txt
    wait
    python3 dashpi.py >> dashpi_log.txt

## So you're having trouble and you need to test everything. These steps seem to be sufficnet as testers

## Check a cron job with:
>> ps aux | grep cron
## Can also check log at tail/var/log/syslog